# PISA Data Downloader
---
If you're not familiar with PISA, it stands for the Programme for International Student Assessment. It's run by the OECD every three years and tests about 600,000 fifteen-year-olds across 80+ countries in math, reading, and science. It's one of the biggest educational assessments in the world, very similar in purpose to the U.S. **National Assessment of Educational Progress (NAEP)**.

The raw data file is a **2 GB SPSS file with 1,200+ columns**. Most of those are individual questionnaire responses we don't need. In this notebook, we'll extract just the variables that matter for predicting math scores, clean them up, and output a tidy CSV.

**What this notebook does:**
1. Downloads the PISA 2022 Student Questionnaire data from the OECD
2. Auto-detects the variable names (they can vary between releases)
3. Cleans and maps codes to readable labels
4. Saves a ready-to-use CSV file

## Imports

In [1]:
import pyreadstat
import pandas as pd
import numpy as np
import os
import zipfile
import urllib.request

## Downloading the Data from OECD

We're pulling the **Student Questionnaire data file** directly from the OECD's PISA 2022 data repository. This is the same file that researchers and governments use worldwide.

It's about **800 MB compressed**. The cell checks if you've already downloaded it so you don't redownload every time you restart the notebook.

In [2]:
ZIP_URL = "https://webfs.oecd.org/pisa2022/STU_QQQ_SPSS.zip"
ZIP_PATH = "STU_QQQ_SPSS.zip"

if not os.path.exists(ZIP_PATH):
    print(f"Downloading PISA 2022 student data from OECD (~800 MB)...")
    print(f"Source: {ZIP_URL}")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print(f"Download complete: {os.path.getsize(ZIP_PATH) / 1e6:.0f} MB")
else:
    print(f"Already downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.0f} MB)")

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    sav_files = [f for f in all_files if f.lower().endswith('.sav')]
    print(f"\nFiles in archive: {all_files}")
    assert len(sav_files) > 0, "No .sav file found in the zip!"
    SAV_FILENAME = sav_files[0]
    
    if not os.path.exists(SAV_FILENAME):
        print(f"Extracting {SAV_FILENAME}...")
        z.extract(SAV_FILENAME)
        print(f"Extracted: {os.path.getsize(SAV_FILENAME) / 1e9:.1f} GB")
    else:
        print(f"Already extracted: {SAV_FILENAME} ({os.path.getsize(SAV_FILENAME) / 1e9:.1f} GB)")

print(f"\nData file ready: {SAV_FILENAME}")

Already downloaded: STU_QQQ_SPSS.zip (682 MB)

Files in archive: ['CY08MSP_STU_QQQ.SAV']
Already extracted: CY08MSP_STU_QQQ.SAV (2.1 GB)

Data file ready: CY08MSP_STU_QQQ.SAV


## Quick Check Column Names

Before loading any actual data, we read **just the metadata**. 

Why do we need this step? Because PISA variable names can differ slightly between data releases. Instead of hardcoding names and hoping they match, we search for each variable we need and match it against what's actually in the file. This makes the notebook robust — it'll work even if OECD tweaks a column name in a future release.

Here are the variables we're looking for and why:

| Variable | What It Is | Use |
|---|---|---|
| `CNT` | Country 3-letter code | Categorical feature |
| `ST004D01T` | Student gender | Categorical feature |
| `MISCED` | Mother's education (ISCED scale) | Categorical feature |
| `FISCED` | Father's education (ISCED scale) | Categorical feature |
| `ESCS` | Economic, Social & Cultural Status index | Key continuous feature |
| `ST013Q01TA` | Books at home (1-6 scale) | Categorical feature |
| `ST011Q06TA` | Internet at home (1=Yes, 2=No) | Categorical feature |
| `MMINS` | Minutes per week studying math | Continuous feature |
| `CLSIZE` | Class size | Continuous feature |
| `PV1MATH` | Math plausible value #1 — **our target** | Target variable |

A quick note on **plausible values**: PISA doesn't give each student a single "score." It generates 10 plausible values (PV1MATH through PV10MATH) that represent the range of proficiency the student likely falls in. For this tutorial, we'll use PV1MATH. In a full research analysis, you'd run the model on all 10 and pool the results.

In [3]:
# Read only metadata — instant, no data loaded
_, meta = pyreadstat.read_sav(SAV_FILENAME, metadataonly=True)
all_cols = meta.column_names
print(f"File has {len(all_cols)} columns total.")

# --- Match our variables against the actual column names ---
def find_col(patterns, all_cols):
    """Return the first column from all_cols matching any pattern (exact first, then substring)."""
    for p in patterns:
        if p in all_cols:
            return p
        matches = [c for c in all_cols if p in c]
        if matches:
            return matches[0]
    return None

# What we want: our_label -> [possible PISA names to search for]
WANTED = {
    'country':           ['CNT'],
    'gender':            ['ST004D01T'],
    'mother_educ_raw':   ['MISCED'],
    'father_educ_raw':   ['FISCED'],
    'escs':              ['ESCS'],
    'books_raw':         ['ST013Q01TA', 'ST013'],
    'internet_raw':      ['ST011Q06TA', 'ST011Q06'],
    'math_minutes_week': ['MMINS', 'LMINS', 'TMINS'],
    'class_size':        ['CLSIZE'],
    'math_score':        ['PV1MATH'],
}

col_map = {}   # pisa_name -> our_label
missing = []
for label, patterns in WANTED.items():
    found = find_col(patterns, all_cols)
    if found:
        col_map[found] = label
    else:
        missing.append(label)

print(f"\nMatched {len(col_map)} of {len(WANTED)} variables:")
for pisa_name, label in col_map.items():
    print(f"  {label:25s} <- {pisa_name}")
if missing:
    print(f"\n  Could not find: {missing}")
    print("  (These will be filled with defaults during cleaning.)")

File has 1278 columns total.

Matched 6 of 10 variables:
  country                   <- CNT
  gender                    <- ST004D01T
  mother_educ_raw           <- MISCED
  father_educ_raw           <- FISCED
  escs                      <- ESCS
  math_score                <- PV1MATH

  Could not find: ['books_raw', 'internet_raw', 'math_minutes_week', 'class_size']
  (These will be filled with defaults during cleaning.)


## Loading the Data

Now we load **only the matched columns** — instead of all 1,200+. This keeps memory usage low and loading fast (usually 20-30 seconds).

In [4]:
usecols = list(col_map.keys())
print(f"Loading {len(usecols)} columns from {SAV_FILENAME}...")
pdf, meta = pyreadstat.read_sav(SAV_FILENAME, usecols=usecols)

# Rename to our standard labels
pdf = pdf.rename(columns=col_map)

print(f"Loaded: {len(pdf):,} rows x {len(pdf.columns)} columns")
print(f"\nColumns: {list(pdf.columns)}")
print(f"\nMissing values per column:")
print(pdf.isnull().sum())
print(f"\nFirst 5 rows:")
pdf.head()

Loading 6 columns from CY08MSP_STU_QQQ.SAV...
Loaded: 613,744 rows x 6 columns

Columns: ['country', 'gender', 'mother_educ_raw', 'father_educ_raw', 'escs', 'math_score']

Missing values per column:
country                0
gender                79
mother_educ_raw    33442
father_educ_raw    45428
escs               25468
math_score             0
dtype: int64

First 5 rows:


,country,gender,mother_educ_raw,father_educ_raw,escs,math_score
0,ALB,1.0,7.0,3.0,1.1112,179.583
1,ALB,2.0,2.0,3.0,-3.0507,308.247
2,ALB,2.0,5.0,3.0,-0.1867,267.514
3,ALB,1.0,2.0,2.0,-3.2198,272.649
4,ALB,1.0,5.0,3.0,-1.0548,435.473


## Cleaning the Data

The raw PISA data uses numeric codes for most categorical variables — we need to map those to readable labels. Here's what we're doing:

**Education levels (MISCED / FISCED):** The PISA codes run from 1 (less than primary) to 10 (doctoral). I'm binning these into 7 broader categories to keep the model manageable.

**Books at home (ST013Q01TA):** Coded 1 through 6, mapping to ranges from "0-10" up to "500+".

**Internet access (ST011Q06TA):** 1 = Yes, 2 = No.

**Missing values:** We drop rows missing the target (math_score) or our most important feature (ESCS). For other variables, we fill with "Unknown" so the model can still use the remaining features for that row.


In [5]:
initial = len(pdf)
pdf = pdf.dropna(subset=['math_score', 'escs'])
print(f"Dropped {initial - len(pdf):,} rows with missing math_score or ESCS")
print(f"Remaining: {len(pdf):,} rows")

educ_map = {
    1: 'Below_secondary', 2: 'Below_secondary', 3: 'Lower_secondary',
    4: 'Upper_secondary', 5: 'Upper_secondary',
    6: 'Post_secondary', 7: 'Short_cycle_tertiary',
    8: 'Bachelors', 9: 'Masters_plus', 10: 'Masters_plus'
}
if 'mother_educ_raw' in pdf.columns:
    pdf['mother_educ'] = pdf['mother_educ_raw'].map(educ_map).fillna('Unknown')
else:
    pdf['mother_educ'] = 'Unknown'
if 'father_educ_raw' in pdf.columns:
    pdf['father_educ'] = pdf['father_educ_raw'].map(educ_map).fillna('Unknown')
else:
    pdf['father_educ'] = 'Unknown'

books_map = {1: '0-10', 2: '11-25', 3: '26-100', 4: '101-200', 5: '201-500', 6: '500+'}
if 'books_raw' in pdf.columns:
    pdf['books_at_home'] = pdf['books_raw'].map(books_map).fillna('Unknown')
else:
    pdf['books_at_home'] = 'Unknown'
    print("Note: books variable not found — filled with 'Unknown'.")

internet_map = {1: 'Yes', 2: 'No'}
if 'internet_raw' in pdf.columns:
    pdf['internet_access'] = pdf['internet_raw'].map(internet_map).fillna('Unknown')
else:
    pdf['internet_access'] = 'Unknown'
    print("Note: internet variable not found — filled with 'Unknown'.")

pdf['gender'] = pdf['gender'].astype(str).str.strip()

if 'math_minutes_week' in pdf.columns:
    pdf['math_minutes_week'] = pd.to_numeric(pdf['math_minutes_week'], errors='coerce').fillna(0)
else:
    pdf['math_minutes_week'] = 0
    print("Note: math_minutes_week not found — filled with 0.")

if 'class_size' in pdf.columns:
    pdf['class_size'] = pd.to_numeric(pdf['class_size'], errors='coerce')
    pdf['class_size'] = pdf['class_size'].fillna(pdf['class_size'].median())
else:
    pdf['class_size'] = 25
    print("Note: class_size not found — filled with default 25.")

final_cols = ['country', 'gender', 'mother_educ', 'father_educ', 'escs',
              'books_at_home', 'internet_access', 'math_minutes_week',
              'class_size', 'math_score']
pdf_clean = pdf[final_cols].copy()

print("Final DataFrame Metrics:")
print(f"Rows:      {len(pdf_clean):,}")
print(f"Columns:   {len(pdf_clean.columns)}")
print(f"Countries: {pdf_clean['country'].nunique()}")
pdf_clean.head(10)

Dropped 25,468 rows with missing math_score or ESCS
Remaining: 588,276 rows
Note: books variable not found — filled with 'Unknown'.
Note: internet variable not found — filled with 'Unknown'.
Note: math_minutes_week not found — filled with 0.
Note: class_size not found — filled with default 25.
Final DataFrame Metrics:
Rows:      588,276
Columns:   10
Countries: 79


,country,gender,mother_educ,father_educ,escs,books_at_home,internet_access,math_minutes_week,class_size,math_score
0,ALB,1.0,Short_cycle_tertiary,Lower_secondary,1.1112,Unknown,Unknown,0,25,179.583
1,ALB,2.0,Below_secondary,Lower_secondary,-3.0507,Unknown,Unknown,0,25,308.247
2,ALB,2.0,Upper_secondary,Lower_secondary,-0.1867,Unknown,Unknown,0,25,267.514
3,ALB,1.0,Below_secondary,Below_secondary,-3.2198,Unknown,Unknown,0,25,272.649
4,ALB,1.0,Upper_secondary,Lower_secondary,-1.0548,Unknown,Unknown,0,25,435.473
5,ALB,2.0,Masters_plus,Masters_plus,1.0855,Unknown,Unknown,0,25,534.112
6,ALB,2.0,Short_cycle_tertiary,Short_cycle_tertiary,-0.7623,Unknown,Unknown,0,25,381.584
7,ALB,1.0,Upper_secondary,Upper_secondary,-1.0237,Unknown,Unknown,0,25,273.196
8,ALB,1.0,Upper_secondary,Upper_secondary,-1.1697,Unknown,Unknown,0,25,355.148
9,ALB,1.0,Upper_secondary,Unknown,0.2857,Unknown,Unknown,0,25,429.818


## Save to CSV

We're saving the cleaned data as a CSV file.

The file should be about **15-25 MB** — dramatically smaller than the 2 GB original because we've cut from 1,200+ columns down to 10 and dropped the SPSS overhead.

In [6]:
OUTPUT_PATH = "pisa_2022_math_clean.csv"
pdf_clean.to_csv(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved: {OUTPUT_PATH} ({size_mb:.1f} MB)")
print(f"  {len(pdf_clean):,} rows x {len(pdf_clean.columns)} columns")
print(f"Original data: ~2 GB -> Cleaned CSV: {size_mb:.1f} MB")

Saved: pisa_2022_math_clean.csv (43.3 MB)
  588,276 rows x 10 columns
Original data: ~2 GB -> Cleaned CSV: 43.3 MB


## Done!

**Files Created:**
- `pisa_2022_math_clean.csv`